# RetainIQ — Phase 2.1: Cleaning Scope & Baseline

## Objective

Translate the Phase 1 audit into an explicit, reproducible cleaning contract **before modifying the raw data**.

### Approved transformations

1. Replace null `Offer` with `No Offer`.
2. Replace null `Internet Type` with `No Internet Service`.
3. Preserve `Churn Category` and `Churn Reason` nulls.
4. Strip leading/trailing whitespace from string columns.
5. Create `is_new_customer = Customer Status == "Joined"`.
6. Preserve one-row-per-customer grain and `Customer ID` uniqueness.
7. Never overwrite the raw source file.

**Phase boundary:** this notebook defines and validates the baseline; transformation happens in later notebooks.

## 1. Environment Setup and Raw Data Ingestion

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\RetainIQ_phase_02_data_cleaning_full\data\telco_raw.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("telco_raw.csv")

candidates = [
    Path(r"C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\RetainIQ_phase_02_data_cleaning_full\data\telco_raw.csv"),
    Path.cwd() / "data" / "telco_raw.csv",
    Path.cwd().parent / "data" / "telco_raw.csv",
    Path.cwd() / "telco_raw.csv",
]

DATA_PATH = next((path for path in candidates if path.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find telco_raw.csv. Current working directory: {Path.cwd()}"
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")

Loaded raw dataset: 7,043 rows × 50 columns


<>:10: SyntaxWarning: invalid escape sequence '\R'
<>:10: SyntaxWarning: invalid escape sequence '\R'
C:\Users\Sahil\AppData\Local\Temp\ipykernel_24548\2528263343.py:10: SyntaxWarning: invalid escape sequence '\R'
  DATA_PATH = Path("C:\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\RetainIQ_phase_02_data_cleaning_full\data\telco_raw.csv")


## 2. Baseline Structure

In [3]:
baseline = pd.Series({
    "Rows": len(df_raw),
    "Columns": len(df_raw.columns),
    "Unique Customer IDs": df_raw["Customer ID"].nunique(),
    "Duplicate Customer IDs": int(df_raw["Customer ID"].duplicated().sum()),
    "Missing Customer IDs": int(df_raw["Customer ID"].isna().sum()),
    "Columns With Missing Values": int(df_raw.isna().any().sum()),
    "Rows With Any Missing Value": int(df_raw.isna().any(axis=1).sum()),
})
baseline

Rows                           7043
Columns                          50
Unique Customer IDs            7043
Duplicate Customer IDs            0
Missing Customer IDs              0
Columns With Missing Values       4
Rows With Any Missing Value    6286
dtype: int64

### Interpretation

The raw file is the source of truth for this phase. The cleaning pipeline must preserve customer
count and customer identity unless a Phase 1 decision explicitly says otherwise.

## 3. Baseline Missingness

In [4]:
missing = df_raw.isna().sum().rename("null_count").to_frame()
missing["null_pct"] = (missing["null_count"] / len(df_raw) * 100).round(2)
missing[missing["null_count"] > 0].sort_values("null_count", ascending=False)

,null_count,null_pct
Churn Reason,5174,73.46
Churn Category,5174,73.46
Offer,3877,55.05
Internet Type,1526,21.67


## 4. Baseline Targeted Fields

In [5]:
for col in ["Offer", "Internet Type", "Churn Category", "Churn Reason", "Customer Status"]:
    print(f"\n{'='*65}\n{col}\n{'='*65}")
    print(df_raw[col].value_counts(dropna=False))


Offer
Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64

Internet Type
Internet Type
Fiber Optic    3035
DSL            1652
NaN            1526
Cable           830
Name: count, dtype: int64

Churn Category
Churn Category
NaN                5174
Competitor          841
Attitude            314
Dissatisfaction     303
Price               211
Other               200
Name: count, dtype: int64

Churn Reason
Churn Reason
NaN                                          5174
Competitor had better devices                 313
Competitor made better offer                  311
Attitude of support person                    220
Don't know                                    130
Competitor offered more data                  117
Competitor offered higher download speeds     100
Attitude of service provider                   94
Price too high                                 78
Product dissatisfaction                        77
Ne

## 5. Pre-Cleaning Validation Gate

Before transforming, confirm that this is the expected Phase 1 source.

This prevents an accidentally different CSV from silently moving through the pipeline.

In [6]:
assert df_raw.shape == (7043, 50)
assert df_raw["Customer ID"].notna().all()
assert df_raw["Customer ID"].is_unique
assert df_raw["Customer Status"].eq("Joined").sum() == 454
assert df_raw["Churn Label"].eq("Yes").sum() == 1869

print("PASS — Expected Phase 1 source confirmed.")

PASS — Expected Phase 1 source confirmed.


## 6. Cleaning Contract

| Area | Rule |
|---|---|
| Offer | Null → `No Offer` |
| Internet Type | Null → `No Internet Service` |
| Churn Category | Preserve null |
| Churn Reason | Preserve null |
| String fields | Strip leading/trailing whitespace |
| New customers | Add boolean `is_new_customer` |
| Customer grain | Preserve one row per unique Customer ID |
| Raw file | Never overwrite |

This contract is the controlling specification for the implementation notebooks.

## 7. Why We Separate Audit From Cleaning

The audit identified issues; this phase now implements documented decisions.

That separation creates a clear lineage:

**Raw source → Audit evidence → Approved rule → Clean dataset → Validation**

This is important for reproducibility and for explaining the project in an interview.

## 8. Notebook 1 Conclusion

The raw dataset is confirmed and the transformation contract is explicit.

**Next:** `02_text_and_categorical_standardization.ipynb`